# Train Image CNN Baseline (PyTorch)

Trains a small CNN on letter-labeled PNGs. Works with either your DB dump (apps/api/data/db_by_letter) or the public dataset (data/hhd_by_letter).

In [ ]:
# Setup
import sys, os, math, random
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


In [ ]:
# Fixed dataset base (no scanning)
BASE = (Path('..') / 'apps/api/data/db_by_letter').resolve()
assert BASE.exists(), f"Missing directory: {BASE}"
LABELS = sorted([p.name for p in BASE.iterdir() if p.is_dir()])
N_CLASSES = len(LABELS)
BASE, N_CLASSES, LABELS[:5]


In [ ]:
# Dataset
class LetterImageDataset(Dataset):
    def __init__(self, files, labels_map, augment=False):
        self.files = list(files)
        self.labels_map = labels_map
        self.augment = augment
    def __len__(self):
        return len(self.files)
    def _augment(self, img):
        # img: (H,W) float32 [0,1] where 1=white background
        if random.random() < 0.5:
            # small rotation
            angle = random.uniform(-10, 10)
            rad = math.radians(angle)
            cos, sin = math.cos(rad), math.sin(rad)
            # naive rotation around center using nearest-neighbor
            h, w = img.shape
            cx, cy = w/2, h/2
            out = np.ones_like(img)
            for y in range(h):
                for x in range(w):
                    rx = int(round(cos*(x-cx) + -sin*(y-cy) + cx))
                    ry = int(round(sin*(x-cx) +  cos*(y-cy) + cy))
                    if 0 <= rx < w and 0 <= ry < h:
                        out[y, x] = img[ry, rx]
            img = out
        return img
    def __getitem__(self, idx):
        p = self.files[idx]
        L = p.parent.name
        y = self.labels_map[L]
        im = Image.open(p).convert('L').resize((64,64))
        arr = np.array(im, dtype=np.float32)/255.0  # [0,1], 1=white
        if self.augment:
            arr = self._augment(arr)
        arr = arr[None, ...]  # (1,64,64)
        x = torch.from_numpy(arr.astype(np.float32))
        return x, torch.tensor(y, dtype=torch.long)

# Build file lists and split
all_files = []
for L in LABELS:
    all_files += sorted((BASE / L).glob('*.png'))
labels_map = {L:i for i,L in enumerate(LABELS)}
random.seed(42)
random.shuffle(all_files)
n = len(all_files); n_train = int(0.8*n); n_val = int(0.1*n)
train_files = all_files[:n_train]
val_files = all_files[n_train:n_train+n_val]
test_files = all_files[n_train+n_val:]
len(train_files), len(val_files), len(test_files)


In [ ]:
# Dataloaders
BATCH = 64
train_ds = LetterImageDataset(train_files, labels_map, augment=True)
val_ds = LetterImageDataset(val_files, labels_map, augment=False)
test_ds = LetterImageDataset(test_files, labels_map, augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH)
test_loader = DataLoader(test_ds, batch_size=BATCH)
next(iter(train_loader))[0].shape


In [ ]:
# Model
class SmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(128*8*8, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.head(self.net(x))

model = SmallCNN(N_CLASSES).to(device)
crit = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def top1(logits, y):
    return (logits.argmax(1) == y).float().mean().item()

hist = {'train_loss': [], 'val_loss': [], 'val_top1': []}
BEST = 0.0
OUT = Path('runs/cnn_baseline_nb'); OUT.mkdir(parents=True, exist_ok=True)
SAVE = OUT / 'model.pt'

for epoch in range(1, 31):
    model.train(); tl, n = 0.0, 0
    for x, y in tqdm(train_loader, desc=f'train {epoch}/30', leave=False):
        x, y = x.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = crit(logits, y)
        loss.backward(); opt.step()
        tl += loss.item()*x.size(0); n += x.size(0)
    hist['train_loss'].append(tl/max(1,n))
    # val
    model.eval(); vl, n, acc = 0.0, 0, 0.0
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc='val', leave=False):
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = crit(logits, y)
            vl += loss.item()*x.size(0); n += x.size(0); acc += top1(logits, y)*x.size(0)
    vl/=max(1,n); acc/=max(1,n); hist['val_loss'].append(vl); hist['val_top1'].append(acc)
    print(f'Epoch {epoch:02d}  train_loss={hist['train_loss'][-1]:.4f}  val_loss={vl:.4f}  val_top1={acc:.3f}')
    if acc > BEST:
        BEST = acc
        torch.save({'model': model.state_dict(), 'labels': LABELS}, SAVE)

print('Best val top1:', BEST, ' Saved:', SAVE)
SAVE


In [ ]:
# Test + Confusion Matrix
from sklearn.metrics import ConfusionMatrixDisplay
model.eval(); all_p, all_y = [], []
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        all_p.append(logits.argmax(1).cpu().numpy()); all_y.append(y.cpu().numpy())
pred = np.concatenate(all_p); true = np.concatenate(all_y)
acc = (pred == true).mean(); print('Test top1:', round(float(acc),3))
cm = confusion_matrix(true, pred, labels=list(range(N_CLASSES)))
disp = ConfusionMatrixDisplay(cm, display_labels=LABELS)
fig, ax = plt.subplots(figsize=(7,7)); disp.plot(ax=ax, xticks_rotation=90, cmap='Blues'); plt.tight_layout(); plt.show()
